# Train From Google Drive (One Click)

This notebook trains the project from a full project zip stored on Google Drive.

Expected zip path:
- `/MyDrive/multi-scale-crop-health/multi-scale-crop-health.zip`

In [ ]:
from google.colab import drive
import os, glob, zipfile, shutil

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/multi-scale-crop-health'
PROJECT_ZIP = os.path.join(DRIVE_ROOT, 'multi-scale-crop-health.zip')
PROJECT_DIR = '/content/multi-scale-crop-health'

print('PROJECT_ZIP exists:', os.path.exists(PROJECT_ZIP))

In [ ]:
if os.path.exists(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)

with zipfile.ZipFile(PROJECT_ZIP, 'r') as zf:
    zf.extractall('/content')

if not os.path.isdir(PROJECT_DIR):
    cands = [p for p in glob.glob('/content/*') if os.path.isdir(p) and 'multi-scale-crop-health' in p.lower()]
    if not cands:
        raise RuntimeError('Could not locate extracted project folder in /content')
    shutil.move(cands[0], PROJECT_DIR)

print('Project directory:', PROJECT_DIR)

In [ ]:
%cd /content/multi-scale-crop-health
!pip -q install -r requirements.txt

import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# Regenerate triplets from your raw folders using current alignment logic
!python preprocessing/align_datasets.py

In [ ]:
# Safety check: verify sampled triplet paths exist (avoid placeholder-image training)
import pandas as pd

df = pd.read_csv('data/triplets/train_triplets.csv')
sample = df.sample(min(len(df), 1500), random_state=42)

ok = 0
for _, r in sample.iterrows():
    ok += int(os.path.exists(r['leaf_path']))
    ok += int(os.path.exists(r['uav_path']))
    ok += int(os.path.exists(r['satellite_path']))

ratio = ok / (len(sample) * 3)
print('Existing-path ratio:', round(ratio, 4))
if ratio < 0.8:
    raise RuntimeError('Too many missing image paths; verify zip contains full data/raw.')

In [ ]:
# Optional Colab stability tweak
import yaml
cfg_path = 'configs/default_config.yaml'
with open(cfg_path, 'r') as f:
    cfg = yaml.safe_load(f)
cfg['batch_size'] = min(int(cfg.get('batch_size', 8)), 4)
cfg['num_workers'] = 2
with open(cfg_path, 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print('batch_size:', cfg['batch_size'], 'num_workers:', cfg['num_workers'])

In [ ]:
!python experiments/run_experiment.py --config configs/exp_full_fusion.yaml

In [ ]:
# Save outputs back to Drive
import datetime

dst_root = '/content/drive/MyDrive/multi-scale-crop-health/outputs'
os.makedirs(dst_root, exist_ok=True)
tag = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
dst = os.path.join(dst_root, f'run_{tag}')
shutil.copytree('/content/multi-scale-crop-health/outputs', dst)
print('Saved outputs to:', dst)